In [ ]:
## 2026.08.20 CODEX PDAC twin of CODEX_hcc/HCC_train_validate_cv_UNIlabel_single.ipynb
## Train + validate on **one** annotated core (s1167 ACQUISITION_ID).
## Cross-dataset notebook: PDAC_train_validate_cv_UNIlabel_all.ipynb
##
## CLI equivalent:
##   conda run --no-capture-output -n SeededNTM python -u \
##     code/CODEX_pdac/PDAC_train_validate_cv_UNIlabel.py --sample Charvill-94_c001_v001_r001_reg001
##
## Hierarchy (three-head):
##   L2  = final_CT          (fine / celltype_level2)
##   L12 = final_sublineage  (intermediate / celltype_level1)
##   L1  = final_lineage     (coarse / celltype_level0)
##
## Preprocess this core first:
##   conda run -n SeededNTM python code/CODEX_pdac/match_codex_cells_with_pixel.py \
##     --acq-id Charvill-94_c001_v001_r001_reg001
##   ACQ_ID=Charvill-94_c001_v001_r001_reg001 bash code/CODEX_pdac/demo_GT_feature_extraction_Single.sh gt
##   ACQ_ID=Charvill-94_c001_v001_r001_reg001 bash code/CODEX_pdac/demo_GT_feature_extraction_Single.sh stardist
## Env: conda activate SeededNTM  (or Hist2Pheno)


## Set env


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("NCRT_CUDA_DEVICE", "0")
import sys as _sys
_pkg_dir = "/home/lingyu/ssd2/Python/Hist2Pheno/code/Hist2Pheno_pkg"
if _pkg_dir not in _sys.path:
    _sys.path.insert(0, _pkg_dir)
from runtime_env import configure_notebook_runtime
_rt_env = configure_notebook_runtime()
NCRT_PHYSICAL_GPU = _rt_env["physical_gpu"]


## Load Hist2Pheno


In [ ]:
import sys
import importlib
import pandas as pd
from pathlib import Path

path = "/home/lingyu/ssd2/Python/"
for _d in (
    f"{path}Hist2Pheno/code/Hist2Pheno_pkg",
    f"{path}Hist2Pheno/code/CODEX_pdac",
):
    if _d not in sys.path:
        sys.path.insert(0, _d)

import base
import plot
import model as model_pkg
importlib.reload(base)
importlib.reload(plot)
importlib.reload(model_pkg)

import uni_label_cv_helpers as uni_nb
importlib.reload(uni_nb)

import PDAC_train_validate_cv_UNIlabel as pdac_cli
importlib.reload(pdac_cli)

from base import first_pth_tensor

print("Imports OK")
print(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')} (physical GPU {NCRT_PHYSICAL_GPU})")


## Define dataset


In [ ]:
## One annotated PDAC TMA core (change SAMPLE to switch)
from uni_label_cv_helpers import RESULT_FIG, make_result_fig

S1167_ROOT = Path(f"{path}Hist2Pheno/data/CODEX/HCC/Michael_data_transfer/s1167")
CASES_ROOT = S1167_ROOT
STARDIST_ROOT = Path(
    os.environ.get(
        "STARDIST_ROOT",
        f"{path}Hist2Pheno/data/CODEX/HCC/StarDist_Segment_pdac/pdac_result",
    )
)

SAMPLE = "Charvill-94_c001_v001_r001_reg001"
therapy_data = SAMPLE
therapy_model = pdac_cli.DEFAULT_THERAPY_MODEL
save_result = pdac_cli.DEFAULT_PER_SAMPLE_SAVE_RESULT
PAN_ORGAN = pdac_cli.PAN_ORGAN

use_spatial_context = True
spatial_k = 8
spatial_mode = "mean"

acq_map = pdac_cli.load_he_acq_map(CASES_ROOT)
if SAMPLE not in acq_map:
    raise KeyError(f"Unknown PDAC ACQUISITION_ID={SAMPLE!r}")
acq_id = acq_map[SAMPLE]

SAMPLE_DIR = CASES_ROOT / SAMPLE
MODEL_DIR = SAMPLE_DIR / therapy_model
cell_coords_path = SAMPLE_DIR / f"{SAMPLE}_cells_with_pixel.csv"
celltype_pixel_stardist_path = SAMPLE_DIR / f"{SAMPLE}_cells_matched_by_stardist.csv"
stardist_raw_path = STARDIST_ROOT / SAMPLE / f"{SAMPLE}_Float_prob0.01_nms_0.3.csv"

print(f"therapy_data (ACQUISITION_ID) = {therapy_data}")
print(f"acq_id (CODEX)                = {acq_id}")
print(f"MODEL_DIR                     = {MODEL_DIR}")
print(f"STARDIST_ROOT                 = {STARDIST_ROOT}")
print(f"pan_organ                     = {PAN_ORGAN}")
print(f"spatial_context               = {use_spatial_context} (k={spatial_k}, mode={spatial_mode!r})")
print(f"GT cells CSV exists           = {cell_coords_path.is_file()}  → {cell_coords_path}")
print(f"StarDist-matched CSV          = {celltype_pixel_stardist_path.is_file()}  → {celltype_pixel_stardist_path}")
print(f"StarDist CSV                  = {stardist_raw_path.is_file()}  → {stardist_raw_path}")
print()
print("Feature extraction (from repo root):")
print(f"  ACQ_ID={SAMPLE} bash code/CODEX_pdac/demo_GT_feature_extraction_Single.sh gt")
print(f"  ACQ_ID={SAMPLE} bash code/CODEX_pdac/demo_GT_feature_extraction_Single.sh stardist")
print("Preprocess cell tables:")
print(f"  python code/CODEX_pdac/match_codex_cells_with_pixel.py --acq-id {SAMPLE}")


In [ ]:
## Result figures under s1167/{SAMPLE}/{therapy_model}/result/
result_fig, _result_dir_path = make_result_fig(
    CASES_ROOT, therapy_data, therapy_model, save_result
)
FIG = RESULT_FIG
BEST_MLP_PATH = MODEL_DIR / "best_mlp_gpu.pt"
print("result_dir =", _result_dir_path)
print("best_mlp  =", BEST_MLP_PATH)


In [ ]:
## CUDA/CPU sanity check + reproducibility
import torch
import random
import numpy as np

print(sys.executable)
print(
    f"torch {torch.__version__} | built with CUDA: {torch.version.cuda} | "
    f"cuda.is_available: {torch.cuda.is_available()}"
)
pdac_cli.setup_cuda(os.environ.get("NCRT_CUDA_DEVICE", "0"))
device = pdac_cli.setup_device(allow_cpu=False)
SEED = 42
pdac_cli.setup_seed(SEED)
print(f"device: {device}")


### Load image feature


In [ ]:
hist_embedding_dir = MODEL_DIR / "ImgEmbeddings_all" / "sc_pth_16_16"
print(hist_embedding_dir)
if hist_embedding_dir.is_dir():
    tensor = first_pth_tensor(hist_embedding_dir)
else:
    print("UNI embedding dir missing — run demo_GT_feature_extraction_Single.sh gt first.")


Example UNI embedding dir for this PDAC core:

`s1167/{ACQUISITION_ID}/project_all_UNI/ImgEmbeddings_all/sc_pth_16_16`

Each `.pth` is typically `torch.Size([1, 1, 1024])`.


## Cell Type Classification Model

Per-sample stratified K-fold on **one** annotated PDAC core (not the 278-core pooled notebook).
CLI twin: `PDAC_train_validate_cv_UNIlabel.py --sample Charvill-94_c001_v001_r001_reg001`.


In [ ]:
## Build per-sample RunContext (mirrors CLI --mode per-sample)

ctx = pdac_cli.RunContext(
    sample=SAMPLE,
    cases_root=CASES_ROOT,
    python_root=Path(path),
    therapy_data=therapy_data,
    therapy_model=therapy_model,
    save_result=save_result,
    device=device,
    seed=SEED,
    match_tolerance=1.0,
    column_rename=dict(pdac_cli.HCC_COLUMN_RENAME),
    force_rebuild_h5ad=False,
    input_dim=None,
    hidden_dims=(1024, 512, 256),
    cv_k=5,
    stratify_target="joint",
    patience=10,
    max_epochs=50,
    train_batch_size=pdac_cli.DEFAULT_TRAIN_BATCH_SIZE,
    resume_from_checkpoints=True,
    ablation_tag=pdac_cli.DEFAULT_ABLATION_TAG,
    hce_w1=1.0, hce_w2=2.0, hce_w12=1.0, hce_w_l12head=1.0, hce_w_l3=1.0, hce_w_l4=1.0,
    build_stardist_h5ad=True,
    acq_id=acq_id,
    stardist_root=STARDIST_ROOT,
    val_selection_metric=pdac_cli.DEFAULT_VAL_SELECTION_METRIC,
    cv_selection_metric=pdac_cli.DEFAULT_CV_SELECTION_METRIC,
    auto_cv_k=True,
    use_spatial_context=use_spatial_context,
    spatial_k=spatial_k,
    spatial_mode=spatial_mode,
)
print(f"ctx.sample_dir = {ctx.sample_dir}")
print(f"matched HE h5ad = {ctx.matched_he_h5ad}")
print(f"matched StarDist h5ad = {ctx.matched_stardist_h5ad}")


## §1 Match UNI embeddings → HE h5ad and prepare CV


In [ ]:
pdac_cli.step_he_h5ad(ctx)
pdac_cli.step_prepare_cv(ctx)
adata = ctx.g["adata"]
print(adata)
print("obs columns:", list(adata.obs.columns))
print("obsm keys:", list(adata.obsm.keys()))


## §2 MLP classifier (stratified K-fold)


In [ ]:
pdac_cli.step_train(ctx)
LP = ctx.g["LP"]
print("BEST_MLP_CHECKPOINT =", ctx.g.get("BEST_MLP_CHECKPOINT"))
print("best_epoch =", LP.get("best_epoch"))
print("val_acc (L2) =", LP.get("val_acc"))
print("val_macro_f1 (L2) =", LP.get("val_macro_f1"))


## §3 Internal validation — L2 / L12 / L1


In [ ]:
## Confusion matrices, per-class F1, spatial pred vs GT (writes under result/)
pdac_cli.step_he_validate(ctx)
print("Internal metrics CSV:", ctx.result_fig("validation_internal_metrics"))


## Predict StarDist


In [ ]:
## Matched StarDist nuclei (with GT AUROC). Builds matched StarDist h5ad if missing.
pdac_cli.step_stardist(ctx)
print("StarDist matched h5ad:", ctx.matched_stardist_h5ad)
print("all_acc =", ctx.g.get("all_acc"))
print("all_macro_f1 =", ctx.g.get("all_macro_f1"))


## Notes

- PDAC demo core: `Charvill-94_c001_v001_r001_reg001` (coverslip c001).
- Training pool of 278 annotated cores is **not** used here; see the `_all` notebook.
- Incomplete_Cases (195 unlabeled) are also only in the `_all` notebook §6.

- Change `SAMPLE` in the dataset cell to run another annotated core.
- Cross-dataset training: `PDAC_train_validate_cv_UNIlabel_all.ipynb`.
- CLI: `python -u code/CODEX_pdac/PDAC_train_validate_cv_UNIlabel.py --sample Charvill-94_c001_v001_r001_reg001`.
- GPU: set `NCRT_CUDA_DEVICE` in the first runtime cell, then **Restart Kernel**.
